# DORA RAG — Phase 1 (v4): Full Corpus Ingestion

Downloads all 16 DORA documents across 3 regulatory levels using the
**EU Publications Office CELLAR REST API** — the official programmatic
backend for EUR-Lex. No browser session, no bot detection, no auth required.

**Run cells top to bottom. Do not skip any cell.**

In [1]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pdfplumber", "requests", "beautifulsoup4", "tqdm",
    "llama-index", "llama-index-llms-groq",
    "llama-index-embeddings-huggingface",
    "llama-index-vector-stores-chroma", "chromadb"
])
print('Dependencies installed.')

Dependencies installed.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [2]:
# Cell 2 — Imports and directory setup
import re
import json
import time
import requests
import pdfplumber
from pathlib import Path

BASE_DIR    = Path("./dora_rag")
DOCS_DIR    = BASE_DIR / "dora_docs"
CLEANED_DIR = BASE_DIR / "dora_cleaned"
CHROMA_DIR  = BASE_DIR / "dora_chroma_db"

for d in [DOCS_DIR, CLEANED_DIR, CHROMA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CELLAR_BASE = "https://publications.europa.eu/resource/cellar"

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent":      "DORA-RAG-Research/1.0 (educational use)",
    "Accept":          "application/pdf, */*",
    "Accept-Language": "en-GB,en;q=0.9",
})

print("Imports OK.")
print(f"Docs dir    : {DOCS_DIR.resolve()}")
print(f"Cleaned dir : {CLEANED_DIR.resolve()}")
print(f"CELLAR API  : {CELLAR_BASE}")

Imports OK.
Docs dir    : /Users/macbook/Downloads/dora_rag/dora_docs
Cleaned dir : /Users/macbook/Downloads/dora_rag/dora_cleaned
CELLAR API  : https://publications.europa.eu/resource/cellar


In [3]:
# Cell 3 — Document registry (all 16 DORA documents across 3 levels)
DORA_DOCUMENTS = [
    # Level 1
    {"name": "L1_DORA_Regulation_2022_2554",           "description": "DORA Regulation (EU) 2022/2554",                          "level": 1, "doc_type": "regulation",          "celex": "32022R2554", "eli": "reg/2022/2554",         "direct_url": None},
    {"name": "L1_DORA_Directive_2022_2556",            "description": "Directive (EU) 2022/2556 — Amending Directive",           "level": 1, "doc_type": "directive",           "celex": "32022L2556", "eli": "dir/2022/2556",         "direct_url": None},
    # Level 2
    {"name": "L2_RTS_ICT_Risk_Management",             "description": "RTS on ICT Risk Management Framework",                   "level": 2, "doc_type": "RTS",                  "celex": "32024R1774", "eli": "reg_del/2024/1774",     "direct_url": None},
    {"name": "L2_RTS_ICT_Incident_Classification",    "description": "RTS on ICT Incident Classification",                     "level": 2, "doc_type": "RTS",                  "celex": "32024R1772", "eli": "reg_del/2024/1772",     "direct_url": None},
    {"name": "L2_RTS_ICT_Incident_Reporting_Process", "description": "RTS on ICT Incident Reporting Process",                   "level": 2, "doc_type": "RTS",                  "celex": "32025R0301", "eli": "reg_del/2025/301",      "direct_url": None},
    {"name": "L2_ITS_ICT_Incident_Reporting",         "description": "ITS on ICT Incident Reporting (templates)",               "level": 2, "doc_type": "ITS",                  "celex": "32025R0302", "eli": "reg_impl/2025/302",     "direct_url": None},
    {"name": "L2_RTS_TLPT",                           "description": "RTS on Threat-Led Penetration Testing (TLPT)",             "level": 2, "doc_type": "RTS",                  "celex": "32025R1190", "eli": "reg_del/2025/1190",     "direct_url": None},
    {"name": "L2_RTS_ICT_Third_Party_Policy",         "description": "RTS on ICT Third-Party Policy",                          "level": 2, "doc_type": "RTS",                  "celex": "32024R1773", "eli": "reg_del/2024/1773",     "direct_url": None},
    {"name": "L2_RTS_Subcontracting",                 "description": "RTS on Subcontracting of ICT Services",                   "level": 2, "doc_type": "RTS",                  "celex": "32025R0532", "eli": "reg_del/2025/532",      "direct_url": None},
    {"name": "L2_ITS_Register_of_Information",        "description": "ITS on Register of Information",                          "level": 2, "doc_type": "ITS",                  "celex": None,         "eli": "reg_impl/2024/2956",    "direct_url": None},
    {"name": "L2_DR_CTPP_Designation_Criteria",       "description": "DR on CTPP Designation Criteria",                        "level": 2, "doc_type": "delegated_regulation", "celex": None,         "eli": None,                    "direct_url": "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=PI_COM:C(2024)896"},
    {"name": "L2_DR_DORA_Oversight_Fees",             "description": "DR on DORA Oversight Fees",                               "level": 2, "doc_type": "delegated_regulation", "celex": None,         "eli": None,                    "direct_url": "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=PI_COM:C(2024)902"},
    {"name": "L2_RTS_Oversight_Conditions",           "description": "RTS on Harmonisation of Oversight Conditions",            "level": 2, "doc_type": "RTS",                  "celex": None,         "eli": "reg_del/2025/295",      "direct_url": None},
    {"name": "L2_RTS_Joint_Examination_Team",         "description": "RTS on Joint Examination Team (JET)",                    "level": 2, "doc_type": "RTS",                  "celex": "32025R0420", "eli": "reg_del/2025/420",      "direct_url": None},
    # Level 3
    {"name": "L3_Guidelines_Costs_and_Losses",        "description": "Guidelines on ICT Incident Costs and Losses (JC 2024-34)","level": 3, "doc_type": "guideline",           "celex": None,         "eli": None,                    "direct_url": "https://www.eiopa.europa.eu/document/download/f92c9ae0-889f-45ed-89bc-7123012e0417_en?filename=JC%202024-34_Guidelines%20on%20costs%20and%20losses_DORA.pdf"},
    {"name": "L3_ESAs_Public_Statement_DORA",         "description": "ESAs Public Statement on DORA Application (JC 2024/99)", "level": 3, "doc_type": "statement",            "celex": None,         "eli": None,                    "direct_url": "https://www.eiopa.europa.eu/document/download/0df4b1d5-245c-4814-b171-ae98d7f8999f_en?filename=JC%202024%2099_ESAs%20Public%20Statement%20on%20DORA%20application.pdf"},
]

print(f"Total documents registered: {len(DORA_DOCUMENTS)}")
for lv in [1, 2, 3]:
    docs = [d for d in DORA_DOCUMENTS if d['level'] == lv]
    print(f"  Level {lv}: {len(docs)} documents")

Total documents registered: 16
  Level 1: 2 documents
  Level 2: 12 documents
  Level 3: 2 documents


In [4]:
# Cell 4 — Download helper functions
def is_valid_pdf(data: bytes) -> bool:
    return len(data) > 4 and data[:4] == b"%PDF"


def download_bytes(url: str, label: str, timeout: int = 60):
    """GET url, validate PDF magic bytes, return bytes or None."""
    try:
        resp = SESSION.get(url, timeout=timeout, allow_redirects=True)
        resp.raise_for_status()
        data = resp.content
        if not is_valid_pdf(data):
            ct = resp.headers.get("Content-Type", "unknown")
            print(f"      ✗ {label}: not a PDF (Content-Type: {ct})")
            return None
        return data
    except requests.HTTPError as e:
        print(f"      ✗ {label}: HTTP {e.response.status_code}")
    except requests.ConnectionError:
        print(f"      ✗ {label}: connection error")
    except requests.Timeout:
        print(f"      ✗ {label}: timeout ({timeout}s)")
    except Exception as e:
        print(f"      ✗ {label}: {e}")
    return None


def fetch_document(doc: dict):
    """
    Try 3 strategies in order:
    1. CELLAR API via CELEX number
    2. CELLAR API via ELI path
    3. Direct URL (for Commission docs and EIOPA guidelines)
    Returns (success: bool, method: str)
    """
    dest = DOCS_DIR / f"{doc['name']}.pdf"

    if dest.exists() and dest.stat().st_size > 20_000:
        print(f"    ✓ cached ({dest.stat().st_size // 1024} KB)")
        return True, "cached"

    data   = None
    method = "failed"

    if doc.get("celex") and data is None:
        url  = f"{CELLAR_BASE}/{doc['celex']}?language=eng&format=pdf"
        print(f"    → CELLAR/CELEX: {url}")
        data = download_bytes(url, "CELLAR CELEX")
        if data:
            method = "cellar_celex"
        time.sleep(0.5)

    if doc.get("eli") and data is None:
        url  = f"{CELLAR_BASE}/eli/{doc['eli']}/oj?language=eng&format=pdf"
        print(f"    → CELLAR/ELI:   {url}")
        data = download_bytes(url, "CELLAR ELI")
        if data:
            method = "cellar_eli"
        time.sleep(0.5)

    if doc.get("direct_url") and data is None:
        url  = doc["direct_url"]
        print(f"    → Direct URL:   {url[:80]}")
        data = download_bytes(url, "direct URL", timeout=90)
        if data:
            method = "direct_url"
        time.sleep(0.5)

    if data:
        dest.write_bytes(data)
        print(f"    ✓ saved {len(data) // 1024} KB → {dest.name} [{method}]")
        return True, method

    return False, "failed"


print("Download functions ready.")

Download functions ready.


In [5]:
# Cell 5 — Download all documents
results = []

for level in [1, 2, 3]:
    level_docs = [d for d in DORA_DOCUMENTS if d["level"] == level]
    print(f"\n{'═'*65}")
    print(f"LEVEL {level} — {len(level_docs)} documents")
    print(f"{'═'*65}")

    for doc in level_docs:
        print(f"\n  [{doc['doc_type'].upper()}] {doc['description']}")
        success, method = fetch_document(doc)

        if not success:
            print(f"    ✗ ALL STRATEGIES FAILED — manual download needed:")
            if doc.get("celex"):
                print(f"      https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:{doc['celex']}")
            if doc.get("eli"):
                print(f"      https://eur-lex.europa.eu/eli/{doc['eli']}/oj")
            if doc.get("direct_url"):
                print(f"      {doc['direct_url'][:80]}")
            print(f"      Save as: {doc['name']}.pdf → {DOCS_DIR}")

        results.append({
            "name":        doc["name"],
            "description": doc["description"],
            "level":       doc["level"],
            "doc_type":    doc["doc_type"],
            "success":     success,
            "method":      method,
        })

succeeded = [r for r in results if r["success"]]
failed    = [r for r in results if not r["success"]]

print(f"\n{'═'*65}")
print(f"DOWNLOAD SUMMARY: {len(succeeded)}/{len(results)} succeeded")
print(f"{'═'*65}")
for r in results:
    icon   = "✓" if r["success"] else "✗"
    method = r['method'] if r['success'] else 'FAILED'
    print(f"  {icon} L{r['level']} [{r['doc_type']:<22}] {r['description'][:38]:<38} [{method}]")

if failed:
    print(f"\n⚠ {len(failed)} document(s) need manual download (see instructions above)")
else:
    print(f"\n✓ All {len(results)} documents downloaded successfully!")


═════════════════════════════════════════════════════════════════
LEVEL 1 — 2 documents
═════════════════════════════════════════════════════════════════

  [REGULATION] DORA Regulation (EU) 2022/2554
    → CELLAR/CELEX: https://publications.europa.eu/resource/cellar/32022R2554?language=eng&format=pdf
      ✗ CELLAR CELEX: HTTP 404
    → CELLAR/ELI:   https://publications.europa.eu/resource/cellar/eli/reg/2022/2554/oj?language=eng&format=pdf
      ✗ CELLAR ELI: HTTP 404
    ✗ ALL STRATEGIES FAILED — manual download needed:
      https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32022R2554
      https://eur-lex.europa.eu/eli/reg/2022/2554/oj
      Save as: L1_DORA_Regulation_2022_2554.pdf → dora_rag/dora_docs

  [DIRECTIVE] Directive (EU) 2022/2556 — Amending Directive
    → CELLAR/CELEX: https://publications.europa.eu/resource/cellar/32022L2556?language=eng&format=pdf
      ✗ CELLAR CELEX: HTTP 404
    → CELLAR/ELI:   https://publications.europa.eu/resource/cellar/eli/dir/

In [7]:
# Cell 6 — Validate all PDFs
def verify_pdf(path: Path) -> dict:
    try:
        with pdfplumber.open(path) as pdf:
            page_count = len(pdf.pages)
            first_text = pdf.pages[0].extract_text() or ""
            return {"valid": True, "pages": page_count,
                    "preview": first_text[:80].replace("\n", " "), "error": None}
    except Exception as e:
        return {"valid": False, "pages": 0, "preview": "", "error": str(e)}


print(f"{'Icon'} {'Lv'} {'Type':<22} {'Document':<38} {'Pages':>5}")
print("─" * 72)

valid_docs = []
for doc in DORA_DOCUMENTS:
    path = DOCS_DIR / f"{doc['name']}.pdf"
    if not path.exists():
        print(f"  ✗  {doc['level']}  {doc['doc_type']:<22} {doc['name'][:38]:<38} {'MISSING':>5}")
        continue
    v    = verify_pdf(path)
    icon = "✓" if v["valid"] else "✗"
    pages = str(v["pages"]) if v["valid"] else "ERR"
    print(f"  {icon}  {doc['level']}  {doc['doc_type']:<22} {doc['name'][:38]:<38} {pages:>5}")
    if v["valid"]:
        valid_docs.append({**doc, "path": path, "pages": v["pages"]})

print(f"\n{len(valid_docs)}/{len(DORA_DOCUMENTS)} documents valid and ready for parsing")

Icon Lv Type                   Document                               Pages
────────────────────────────────────────────────────────────────────────
  ✓  1  regulation             L1_DORA_Regulation_2022_2554              79
  ✓  1  directive              L1_DORA_Directive_2022_2556               11
  ✓  2  RTS                    L2_RTS_ICT_Risk_Management                37
  ✓  2  RTS                    L2_RTS_ICT_Incident_Classification         9
  ✓  2  RTS                    L2_RTS_ICT_Incident_Reporting_Process      5
  ✓  2  ITS                    L2_ITS_ICT_Incident_Reporting             44
  ✓  2  RTS                    L2_RTS_TLPT                               21
  ✓  2  RTS                    L2_RTS_ICT_Third_Party_Policy              9
  ✓  2  RTS                    L2_RTS_Subcontracting                      7
  ✓  2  ITS                    L2_ITS_Register_of_Information            43
  ✓  2  delegated_regulation   L2_DR_CTPP_Designation_Criteria           12
  ✓  2  delegat

In [8]:
# Cell 7 — Text processing functions
def parse_pdf(pdf_path: Path) -> dict:
    pages, parts = [], []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text(x_tolerance=2, y_tolerance=2)
            if text:
                pages.append({"page_num": i + 1, "text": text})
                parts.append(text)
    return {
        "source_file": pdf_path.name,
        "page_count":  len(pages),
        "pages":       pages,
        "full_text":   "\n\n".join(parts),
    }


def clean_legal_text(text: str) -> str:
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    text = re.sub(r'\n\s*\d{1,3}\s*\n', '\n', text)
    text = re.sub(r'Official Journal.*?\n', '', text, flags=re.IGNORECASE)
    text = re.sub(r'EN\s+L \d+\/\d+.*?\n', '', text)
    text = text.replace('\u00ad', '')
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def extract_article_map(text: str) -> list:
    pattern  = re.compile(r'(Article\s+(\d+)\s*\n([^\n]+))', re.IGNORECASE)
    articles = []
    for match in pattern.finditer(text):
        articles.append({
            "article_num": int(match.group(2)),
            "title":       match.group(3).strip(),
            "char_start":  match.start(),
        })
    for i in range(len(articles) - 1):
        articles[i]["char_end"] = articles[i + 1]["char_start"]
    if articles:
        articles[-1]["char_end"] = len(text)
    return articles


print("Text processing functions ready.")

Text processing functions ready.


In [9]:
# Cell 8 — Parse, clean and extract articles from all valid documents
parsed_docs = {}

for doc in valid_docs:
    print(f"  L{doc['level']} {doc['doc_type']}: {doc['name']}")
    try:
        parsed      = parse_pdf(doc["path"])
        cleaned     = clean_legal_text(parsed["full_text"])
        article_map = extract_article_map(cleaned)

        parsed["level"]        = doc["level"]
        parsed["doc_type"]     = doc["doc_type"]
        parsed["description"]  = doc["description"]
        parsed["celex"]        = doc.get("celex")
        parsed["cleaned_text"] = cleaned
        parsed["article_map"]  = article_map

        parsed_docs[doc["name"]] = parsed

        art_info = ""
        if article_map:
            art_info = f" | Art {article_map[0]['article_num']}–{article_map[-1]['article_num']}"
        print(f"    ✓ {parsed['page_count']} pages, {len(cleaned):,} chars, {len(article_map)} articles{art_info}")

    except Exception as e:
        print(f"    ✗ Parse error: {e}")

print(f"\nProcessed {len(parsed_docs)}/{len(valid_docs)} documents")

  L1 regulation: L1_DORA_Regulation_2022_2554
    ✓ 79 pages, 304,419 chars, 66 articles | Art 101–64
  L1 directive: L1_DORA_Directive_2022_2556
    ✓ 11 pages, 37,960 chars, 12 articles | Art 1–11
  L2 RTS: L2_RTS_ICT_Risk_Management
    ✓ 37 pages, 123,338 chars, 42 articles | Art 1–42
  L2 RTS: L2_RTS_ICT_Incident_Classification
    ✓ 9 pages, 31,277 chars, 13 articles | Art 1–13
  L2 RTS: L2_RTS_ICT_Incident_Reporting_Process
    ✓ 5 pages, 16,759 chars, 7 articles | Art 1–7
  L2 ITS: L2_ITS_ICT_Incident_Reporting
    ✓ 44 pages, 96,582 chars, 9 articles | Art 1–9
  L2 RTS: L2_RTS_TLPT
    ✓ 21 pages, 30,899 chars, 0 articles
  L2 RTS: L2_RTS_ICT_Third_Party_Policy
    ✓ 9 pages, 29,979 chars, 4 articles | Art 1–4
  L2 RTS: L2_RTS_Subcontracting
    ✓ 7 pages, 25,097 chars, 7 articles | Art 1–7
  L2 ITS: L2_ITS_Register_of_Information
    ✓ 43 pages, 103,606 chars, 8 articles | Art 1–9
  L2 delegated_regulation: L2_DR_CTPP_Designation_Criteria
    ✓ 12 pages, 31,779 chars, 7 artic

In [10]:
# Cell 9 — Save cleaned text and manifest
manifest = []

for name, doc in parsed_docs.items():
    cleaned     = doc["cleaned_text"]
    article_map = doc["article_map"]

    txt_path = CLEANED_DIR / f"{name}.txt"
    txt_path.write_text(cleaned, encoding="utf-8")

    map_path = None
    if article_map:
        map_path = CLEANED_DIR / f"{name}_article_map.json"
        map_path.write_text(json.dumps(article_map, indent=2), encoding="utf-8")

    manifest.append({
        "name":             name,
        "level":            doc["level"],
        "doc_type":         doc["doc_type"],
        "description":      doc["description"],
        "celex":            doc.get("celex"),
        "page_count":       doc["page_count"],
        "cleaned_chars":    len(cleaned),
        "article_count":    len(article_map),
        "txt_file":         str(txt_path),
        "article_map_file": str(map_path) if map_path else None,
    })
    print(f"  ✓ {name}")

manifest_path = CLEANED_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"\n✓ Manifest saved: {manifest_path}")

  ✓ L1_DORA_Regulation_2022_2554
  ✓ L1_DORA_Directive_2022_2556
  ✓ L2_RTS_ICT_Risk_Management
  ✓ L2_RTS_ICT_Incident_Classification
  ✓ L2_RTS_ICT_Incident_Reporting_Process
  ✓ L2_ITS_ICT_Incident_Reporting
  ✓ L2_RTS_TLPT
  ✓ L2_RTS_ICT_Third_Party_Policy
  ✓ L2_RTS_Subcontracting
  ✓ L2_ITS_Register_of_Information
  ✓ L2_DR_CTPP_Designation_Criteria
  ✓ L2_DR_DORA_Oversight_Fees
  ✓ L2_RTS_Oversight_Conditions
  ✓ L2_RTS_Joint_Examination_Team
  ✓ L3_Guidelines_Costs_and_Losses
  ✓ L3_ESAs_Public_Statement_DORA

✓ Manifest saved: dora_rag/dora_cleaned/manifest.json


In [11]:
# Cell 10 — Final summary
total_chars    = sum(e["cleaned_chars"] for e in manifest)
total_pages    = sum(e["page_count"] for e in manifest)
total_articles = sum(e["article_count"] for e in manifest)

print("\n" + "═" * 65)
print("PHASE 1 COMPLETE")
print("═" * 65)
print(f"\n{'Lv'} {'Type':<22} {'Document':<38} {'Pg':>4} {'Art':>4}")
print("─" * 72)

for level in [1, 2, 3]:
    entries = [e for e in manifest if e["level"] == level]
    if entries:
        print(f"── Level {level} {'─'*60}")
    for e in entries:
        print(f"  {e['level']}  {e['doc_type']:<22} {e['name'][:38]:<38} {e['page_count']:>4} {e['article_count']:>4}")

print("─" * 72)
print(f"  {'TOTAL':<63} {total_pages:>4} {total_articles:>4}")
print(f"\n  Documents  : {len(manifest)}/{len(DORA_DOCUMENTS)}")
print(f"  Total text : {total_chars:,} characters")

still_failed = [r for r in results if not r["success"]]
if still_failed:
    print(f"\n  ⚠ {len(still_failed)} documents not downloaded:")
    for f in still_failed:
        print(f"    L{f['level']} — {f['description']}")

print("\nNext step → open 02_chunking.ipynb")


═════════════════════════════════════════════════════════════════
PHASE 1 COMPLETE
═════════════════════════════════════════════════════════════════

Lv Type                   Document                                 Pg  Art
────────────────────────────────────────────────────────────────────────
── Level 1 ────────────────────────────────────────────────────────────
  1  regulation             L1_DORA_Regulation_2022_2554             79   66
  1  directive              L1_DORA_Directive_2022_2556              11   12
── Level 2 ────────────────────────────────────────────────────────────
  2  RTS                    L2_RTS_ICT_Risk_Management               37   42
  2  RTS                    L2_RTS_ICT_Incident_Classification        9   13
  2  RTS                    L2_RTS_ICT_Incident_Reporting_Process     5    7
  2  ITS                    L2_ITS_ICT_Incident_Reporting            44    9
  2  RTS                    L2_RTS_TLPT                              21    0
  2  RTS          